# Tutorial 2 — Spatial associations with spin tests

**Goal:** test each cell-type–IDP association while preserving cortical spatial autocorrelation in the null model. This follows the logic of the [BrainSpace spin tutorial](https://brainspace.readthedocs.io/en/latest/python_doc/auto_examples/plot_tutorial3.html).

<!-- github-visual-preview -->
![Tutorial visual preview](figures/02_spin_test.png)

*Deterministic preview generated from the released BN map and example IDPs. Running the notebook redraws the figure from the current inputs.*


In [ ]:
from pathlib import Path
import sys
import pandas as pd

HERE = Path.cwd() if (Path.cwd() / 'tutorial_utils.py').exists() else Path.cwd() / 'tutorials'
sys.path.insert(0, str(HERE))
from tutorial_utils import find_repo_root, load_prepared_or_example, align_and_validate
from HomoloMap.stats import SpinTest
from HomoloMap.utils import run_spin_correlations

ROOT = find_repo_root()
N_SPINS = 100  # use >=1,000 for scientific analysis
SEED = 42
X, Y = load_prepared_or_example(ROOT, level='subclass')
X, Y = align_and_validate(X, Y, require_complete_bn=True)

## Generate one reusable spatial null model

The same rotations are reused across cell types. Benjamini–Hochberg correction is applied separately within each IDP across the selected cell-type resolution.

In [ ]:
spinner = SpinTest(atlas='BN', n_spins=N_SPINS, method='Alexander-Bloch', seed=SEED)
spin = run_spin_correlations(
    X, Y, spinner, metric='pearsonr', FDR='fdr_bh', n_jobs=1,
    composition_transform='none',
)
spin.head()

In [ ]:
idp = Y.columns[0]
r_col = f'{idp}_ratio_spin_r'
q_col = f'{idp}_ratio_spin_p_adj'
display(spin[[r_col, q_col]].sort_values(q_col).head(10))

## Interpretation

The coefficient gives direction and magnitude; the spin p-value evaluates spatial correspondence. FDR significance is not evidence of causality. Define the comparison family before examining results.

In [ ]:
OUTPUT = ROOT / 'tutorial_outputs'
OUTPUT.mkdir(exist_ok=True)
spin.to_csv(OUTPUT / 'spin_results_ratio.csv')

<!-- tutorial-visual-summary -->
### Visualize spatial associations
Bars show spatially corrected correlations for one IDP. Filled circles mark associations passing the selected FDR threshold.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plot_data = spin[[r_col, q_col]].sort_values(r_col)
colors = np.where(plot_data[q_col] < 0.05, '#d1495b', '#9db7c4')
fig, ax = plt.subplots(figsize=(7, max(4, 0.22 * len(plot_data))))
ax.barh(plot_data.index, plot_data[r_col], color=colors, edgecolor='none')
ax.axvline(0, color='0.25', linewidth=0.8)
ax.set(xlabel='Pearson correlation', ylabel='Cell type',
       title=f'Spin-test associations with {idp}')
sns.despine()
fig.tight_layout()
